# Notebook 2 — Tiền xử lý văn bản tiếng Việt cho bài toán gom cụm

## 1. Bối cảnh

Sau Notebook 1, dữ liệu đã được hợp nhất, loại trùng và lọc các bài viết quá ngắn để tạo thành file sạch `bai_test_do_an_main_clean.csv`. Tuy nhiên, để phục vụ tốt cho bài toán **gom cụm văn bản báo chí tiếng Việt**, dữ liệu cần được tiền xử lý sâu hơn ở mức ngôn ngữ.

Văn bản tiếng Việt có nhiều đặc thù như từ ghép đa âm tiết, nhiều từ chức năng, từ báo chí lặp lại thường xuyên và mức độ chồng lấn chủ đề giữa các chuyên mục. Vì vậy, bước tiền xử lý có ảnh hưởng trực tiếp đến chất lượng biểu diễn văn bản và khả năng tách cụm ở các notebook sau.

## 2. Mục tiêu của notebook

Notebook này tập trung xây dựng các cột văn bản đầu vào cho bài toán gom cụm, gồm:

- Tạo cột văn bản chính từ `title`, `description` và `content`.
- Chuẩn hóa Unicode, chữ thường, ký tự đặc biệt và khoảng trắng.
- Tách từ tiếng Việt bằng PyVi để giữ các cụm từ đa âm tiết.
- Xây dựng bộ stopwords tùy chỉnh dựa trên corpus báo chí.
- Tạo cột văn bản cho nhánh lexical như TF-IDF và BM25.
- Tạo cột văn bản sạch tự nhiên cho nhánh semantic embedding.

## 3. Lưu ý về bản chất bài toán

Đây là bài toán **gom cụm văn bản**, không phải bài toán phân loại.

Cột `category_clean` chỉ được giữ lại để:

- mô tả phân bố dữ liệu,
- kiểm tra tính cân bằng tương đối của bộ dữ liệu,
- và có thể dùng làm tham chiếu khi tính các chỉ số đánh giá ngoài như ARI, NMI hoặc Purity.

Cột này không được dùng để huấn luyện mô hình và không được xem là nhãn dự đoán.

## 4. Đầu ra mong đợi

Sau notebook này, dữ liệu sẽ được lưu thành file:

`bai_test_do_an_preprocessed.csv`

File đầu ra gồm các cột văn bản chính:

- `cluster_text`: văn bản ghép gốc.
- `cluster_text_clean`: văn bản đã chuẩn hóa nhẹ.
- `cluster_text_segmented`: văn bản đã tách từ tiếng Việt.
- `cluster_text_lexical`: văn bản đã tách từ và loại stopwords, dùng cho TF-IDF/BM25.
- `cluster_text_semantic`: văn bản sạch tự nhiên, dùng cho semantic embedding ở Notebook 4.

In [1]:
# Cell 2: import thư viện và khai báo đường dẫn dữ liệu cho Notebook 2

import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

PROJECT_ROOT = Path(r"D:\PROJECT")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
REPORT_DIR = OUTPUT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "news_clean.csv"
OUTPUT_PREPROCESSED = PROCESSED_DIR / "news_preprocessed.csv"
STOPWORDS_PATH = REPORT_DIR / "refined_stopwords.txt"

print("File đầu vào:", INPUT_PATH)
print("Tồn tại:", INPUT_PATH.exists())

print("\nFile tiền xử lý sẽ lưu tại:")
print(OUTPUT_PREPROCESSED)

print("\nFile stopwords sẽ lưu tại:")
print(STOPWORDS_PATH)

File đầu vào: D:\PROJECT\data\processed\news_clean.csv
Tồn tại: True

File tiền xử lý sẽ lưu tại:
D:\PROJECT\data\processed\news_preprocessed.csv

File stopwords sẽ lưu tại:
D:\PROJECT\outputs\reports\refined_stopwords.txt


In [2]:
# Cell 3: đọc file dữ liệu sạch và kiểm tra nhanh cấu trúc đầu vào
# Cell này giúp xác nhận file đầu vào đã đúng và sẵn sàng cho bước tạo văn bản chính phục vụ NLP.

df = pd.read_csv(INPUT_PATH)

print("Kích thước dữ liệu:", df.shape)
print("\nCác cột hiện có:")
print(df.columns.tolist())

print("\nSố giá trị thiếu theo cột:")
display(df.isna().sum().to_frame("missing_count"))

display(df.head(3))

Kích thước dữ liệu: (8727, 11)

Các cột hiện có:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'raw_text', 'content_word_count', 'word_count']

Số giá trị thiếu theo cột:


,missing_count
doc_id,0
source,0
category_clean,0
title,0
description,0
content,0
published_date,0
url,0
raw_text,0
content_word_count,0


,doc_id,source,category_clean,title,description,content,published_date,url,raw_text,content_word_count,word_count
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,1036,1096
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,Meta đưa AI vào Messenger trả lời khách hàng 2...,565,615
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,313,366


In [3]:
# Cell 4: tạo cột văn bản chính cho bài toán NLP
# Cell này ghép title, description và content thành một trường văn bản thống nhất để dùng cho các bước chuẩn hóa và tách từ phía sau.

df_work = df.copy()

df_work["cluster_text"] = (
    df_work["title"].astype(str).str.strip() + " . " +
    df_work["description"].astype(str).str.strip() + " . " +
    df_work["content"].astype(str).str.strip()
)

print("Kích thước dữ liệu làm việc:", df_work.shape)
print("\nCác cột hiện có sau khi tạo cluster_text:")
print(df_work.columns.tolist())

display(
    df_work[["doc_id", "category_clean", "title", "description", "content", "cluster_text"]].head(2)
)

Kích thước dữ liệu làm việc: (8727, 12)

Các cột hiện có sau khi tạo cluster_text:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'raw_text', 'content_word_count', 'word_count', 'cluster_text']


,doc_id,category_clean,title,description,content,cluster_text
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,Meta đưa AI vào Messenger trả lời khách hàng 2...


In [4]:
# Cell 5: định nghĩa hàm chuẩn hóa văn bản mức nhẹ
# Cell này tạo bước làm sạch cơ sở để giảm nhiễu nhưng vẫn giữ lại phần lớn tín hiệu ngôn ngữ cần cho gom cụm.

def normalize_vietnamese_text(text):
    text = str(text)

    # Chuẩn hóa Unicode
    text = unicodedata.normalize("NFC", text)

    # Bỏ HTML nếu có
    text = re.sub(r"<[^>]+>", " ", text)

    # Bỏ URL nếu có
    text = re.sub(r"http[s]?://\S+|www\.\S+", " ", text)

    # Chuẩn hóa xuống chữ thường
    text = text.lower()

    # Thay các ký tự xuống dòng, tab bằng khoảng trắng
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Giữ chữ, số, khoảng trắng và một số dấu câu cơ bản để tránh làm dính từ quá sớm
    text = re.sub(r"[^0-9a-zA-ZÀ-Ỹà-ỹ\s\.\,\!\?\:\;\-\%\/]", " ", text)

    # Chuẩn hóa nhiều khoảng trắng liên tiếp
    text = re.sub(r"\s+", " ", text).strip()

    return text

print("Đã sẵn sàng hàm normalize_vietnamese_text()")

Đã sẵn sàng hàm normalize_vietnamese_text()


In [5]:
# Cell 6: áp dụng chuẩn hóa văn bản cho cột cluster_text
# Cell này tạo ra cột cluster_text_clean để dùng làm nền cho bước tách từ tiếng Việt ở cell sau.

df_work["cluster_text_clean"] = df_work["cluster_text"].apply(normalize_vietnamese_text)

print("Kích thước dữ liệu sau khi tạo cluster_text_clean:", df_work.shape)

display(
    df_work[["doc_id", "category_clean", "cluster_text", "cluster_text_clean"]].head(2)
)

Kích thước dữ liệu sau khi tạo cluster_text_clean: (8727, 13)


,doc_id,category_clean,cluster_text,cluster_text_clean
0,1,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,làn sóng máy tính lượng tử sau cơn sốt ai . qu...
1,2,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả lời khách hàng 2...


In [6]:
# Cell 7: cài đặt/import PyVi và tách từ tiếng Việt
# Cell này thực hiện bước quan trọng của tiền xử lý tiếng Việt: nhận diện từ ghép và tạo cột cluster_text_segmented
# để chuẩn bị cho các nhánh biểu diễn lexical như TF-IDF và BM25.

try:
    from pyvi import ViTokenizer
    print("Đã import pyvi thành công.")
except ImportError:
    !pip -q install pyvi
    from pyvi import ViTokenizer
    print("Đã cài đặt và import pyvi thành công.")

import time

def segment_vietnamese_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    return ViTokenizer.tokenize(text)

print("\nĐang tiến hành tách từ cho toàn bộ văn bản... (quá trình này có thể mất 1-2 phút)")
start_time = time.time()

df_work["cluster_text_segmented"] = df_work["cluster_text_clean"].apply(segment_vietnamese_text)

elapsed_time = time.time() - start_time

print(f"Hoàn thành tách từ trong: {elapsed_time:.2f} giây")
print("Kích thước dữ liệu hiện tại:", df_work.shape)

print("\nSo sánh văn bản trước và sau khi tách từ (dòng 0):")
print("- Gốc sạch :", df_work.loc[0, "cluster_text_clean"][:200], "...")
print("- Tách từ  :", df_work.loc[0, "cluster_text_segmented"][:200], "...")

display(
    df_work[["doc_id", "category_clean", "cluster_text_segmented"]].head(2)
)

Đã import pyvi thành công.

Đang tiến hành tách từ cho toàn bộ văn bản... (quá trình này có thể mất 1-2 phút)
Hoàn thành tách từ trong: 49.18 giây
Kích thước dữ liệu hiện tại: (8727, 14)

So sánh văn bản trước và sau khi tách từ (dòng 0):
- Gốc sạch : làn sóng máy tính lượng tử sau cơn sốt ai . quý 1-2026, thế giới chứng kiến loạt công ty vi tính lượng tử liên tiếp niêm yết trên các sàn chứng khoán mỹ, được xem là tín hiệu rõ rệt cho thấy công nghệ ...
- Tách từ  : làn_sóng máy_tính lượng_tử sau cơn_sốt ai . quý 1 - 2026 , thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết trên các sàn chứng_khoán mỹ , được xem là tín_hiệu rõ_rệt cho thấy công_ ...


,doc_id,category_clean,cluster_text_segmented
0,1,cong_nghe,làn_sóng máy_tính lượng_tử sau cơn_sốt ai . qu...
1,2,cong_nghe,meta đưa ai vào messenger trả_lời khách_hàng 2...


### Nhận xét nhanh sau bước tách từ tiếng Việt

Bước tách từ bằng `PyVi` đã hoạt động đúng như kỳ vọng. Các cụm từ đa âm tiết quan trọng như `làn_sóng`, `máy_tính`, `lượng_tử`, `chứng_khoán`, `tín_hiệu` đã được giữ lại dưới dạng token ghép, giúp bảo toàn ý nghĩa ngữ nghĩa tốt hơn so với việc tách rời từng âm tiết.

Đây là tín hiệu tích cực cho các bước biểu diễn văn bản theo hướng lexical như **TF-IDF** và **BM25**, vì các mô hình này rất nhạy với chất lượng token đầu vào.

In [7]:
# Cell 9: tạo danh sách ứng viên stopwords từ chính corpus đã tách từ
# Cell này dùng document frequency và độ phủ theo chuyên mục để tìm các token quá phổ biến,
# từ đó hỗ trợ xây dựng bộ stopwords phù hợp riêng cho dữ liệu báo chí của đồ án.

from collections import Counter, defaultdict

token_doc_freq = Counter()
token_category_set = defaultdict(set)

for _, row in df_work[["category_clean", "cluster_text_segmented"]].iterrows():
    category = row["category_clean"]
    text = row["cluster_text_segmented"]

    if not isinstance(text, str) or text.strip() == "":
        continue

    tokens = set(text.split())

    for token in tokens:
        token_doc_freq[token] += 1
        token_category_set[token].add(category)

candidate_rows = []
n_docs = len(df_work)
n_categories = df_work["category_clean"].nunique()

for token, dfreq in token_doc_freq.items():
    doc_ratio = dfreq / n_docs
    category_coverage = len(token_category_set[token]) / n_categories

    candidate_rows.append({
        "token": token,
        "doc_freq": dfreq,
        "doc_ratio": round(doc_ratio, 4),
        "category_count": len(token_category_set[token]),
        "category_coverage": round(category_coverage, 4),
        "token_length": len(token)
    })

candidate_stopwords = pd.DataFrame(candidate_rows)

# lọc các token quá phổ biến và phủ rộng nhiều chuyên mục
candidate_stopwords = candidate_stopwords[
    (candidate_stopwords["doc_ratio"] >= 0.20) &
    (candidate_stopwords["category_coverage"] >= 0.80)
].copy()

# loại bớt token quá ngắn hoặc chỉ là dấu câu
candidate_stopwords = candidate_stopwords[
    candidate_stopwords["token"].str.contains(r"[a-zA-ZÀ-Ỹà-ỹ0-9_]", regex=True)
]

candidate_stopwords = candidate_stopwords.sort_values(
    by=["doc_ratio", "category_coverage", "doc_freq"],
    ascending=False
).reset_index(drop=True)

print("Số lượng ứng viên stopwords:", len(candidate_stopwords))
display(candidate_stopwords.head(100))

Số lượng ứng viên stopwords: 201


,token,doc_freq,doc_ratio,category_count,category_coverage,token_length
0,và,8656,0.9919,6,1.0,2
1,trong,8349,0.9567,6,1.0,5
2,của,8250,0.9453,6,1.0,3
3,các,8167,0.9358,6,1.0,3
4,cho,8155,0.9345,6,1.0,3
...,...,...,...,...,...,...
95,2026,2820,0.3231,6,1.0,4
96,nếu,2807,0.3216,6,1.0,3
97,đồng_thời,2804,0.3213,6,1.0,9
98,giữa,2795,0.3203,6,1.0,4


In [8]:
# Cell 10: xem kỹ nhóm ứng viên stopwords phổ biến nhất để chốt danh sách tùy chỉnh
# Cell này giúp quan sát trực tiếp các token phủ rộng toàn corpus trước khi quyết định loại bỏ khỏi nhánh lexical.

top_n = 120

candidate_preview = candidate_stopwords.head(top_n).copy()

print(f"Hiển thị {top_n} ứng viên stopwords đầu tiên:")
display(candidate_preview)

print("\nDanh sách token top đầu:")
print(candidate_preview["token"].tolist())

Hiển thị 120 ứng viên stopwords đầu tiên:


,token,doc_freq,doc_ratio,category_count,category_coverage,token_length
0,và,8656,0.9919,6,1.0,2
1,trong,8349,0.9567,6,1.0,5
2,của,8250,0.9453,6,1.0,3
3,các,8167,0.9358,6,1.0,3
4,cho,8155,0.9345,6,1.0,3
...,...,...,...,...,...,...
115,5,2570,0.2945,6,1.0,1
116,thông_tin,2561,0.2935,6,1.0,9
117,tiếp_tục,2561,0.2935,6,1.0,8
118,khiến,2531,0.2900,6,1.0,5



Danh sách token top đầu:
['và', 'trong', 'của', 'các', 'cho', 'được', 'với', 'là', 'có', 'đến', 'đã', 'để', 'từ', 'đó', 'không', 'khi', 'này', 'theo', 'tại', 'nhiều', 'một', 'về', 'trên', 'người', 'ở', 'vào', 'cũng', 'những', 'ngày', 'như', 'ra', 'hơn', 'còn', 'sau', 'năm', 'việc', 'sẽ', 'chỉ', 'trước', 'đây', 'biết', 'lại', 'đang', 'sự', 'phải', 'cao', 'mới', 'nhưng', 'nhất', 'có_thể', 'cùng', 'mà', 'qua', 'làm', '3', 'do', '2', 'lớn', '1', 'vẫn', 'lên', 'đi', 'hai', 'bị', 'cần', 'đặc_biệt', 'rất', 'tổ_chức', 'khác', 'ông', 'hay', 'cả', 'tạo', 'điểm', 'thời_gian', 'giúp', 'gần', 'hoặc', 'tới', 'thấy', 'tp', 'nam', 'số', 'nhà', 'chưa', 'đưa', 'từng', 'việt_nam', 'khoảng', 'nên', 'nước', 'phát_triển', 'lần', 'mang', 'hoạt_động', '2026', 'nếu', 'đồng_thời', 'giữa', 'bên', '2025', '4', 'bằng', 'thêm', 'nói', 'rằng', 'bộ', 'tuy_nhiên', 'đầu', 'tháng', '10', 'ngay', 'mỗi', 'nguyễn', 'sử_dụng', '5', 'thông_tin', 'tiếp_tục', 'khiến', 'vừa']


In [9]:
# Cell 11: chốt bộ stopwords tùy chỉnh và tạo cột văn bản lexical
# Cell này loại bỏ các token chức năng, token báo chí phổ biến và các từ có khả năng gây nhiễu khi gom cụm.

custom_stopwords = {
    # Nhóm từ chức năng phổ biến
    "và", "trong", "của", "các", "cho", "được", "với", "là", "có", "đến",
    "đã", "để", "từ", "đó", "không", "khi", "này", "theo", "tại", "nhiều",
    "một", "về", "trên", "ở", "vào", "cũng", "những", "như", "ra", "hơn",
    "còn", "sau", "năm", "việc", "sẽ", "chỉ", "trước", "đây", "lại", "đang",
    "sự", "phải", "nhưng", "mà", "qua", "do", "vẫn", "lên", "đi", "hai",
    "bị", "cần", "rất", "khác", "hay", "cả", "gần", "hoặc", "tới", "chưa",
    "từng", "khoảng", "nên", "lần", "nếu", "giữa", "bên", "bằng", "thêm",
    "rằng", "tuy_nhiên", "đầu", "ngay", "mỗi", "vừa", "gồm", "gồm_có",

    # Nhóm thời gian, số liệu chung
    "ngày", "tháng", "tuần", "quý", "năm", "hôm_nay", "hôm_qua",
    "sáng", "chiều", "tối", "nay", "hiện", "hiện_nay",
    "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
    "2022", "2023", "2024", "2025", "2026",

    # Nhóm từ báo chí xuất hiện rộng nhưng ít phân biệt chủ đề
    "người", "cho_biết", "biết", "tiếp_tục", "đồng_thời",
    "khiến", "xem", "thấy", "nói", "chia_sẻ", "nhận_định", "đánh_giá",
    "đề_xuất", "khẳng_định", "cho_rằng", "liên_quan", "diễn_ra",
    "tổ_chức", "hoạt_động", "nội_dung", "vấn_đề", "trường_hợp",
    "khu_vực", "địa_phương", "đơn_vị", "cơ_quan",

    # Nhóm địa danh/quốc gia quá phổ biến trong corpus báo chí Việt Nam
    "việt_nam", "việt", "nam", "tp", "hcm", "tp_hcm", "tphcm",
    "hà_nội", "đà_nẵng", "mỹ", "trung_quốc",

    # Nhóm ký hiệu / từ liệt kê không mang ý nghĩa chủ đề
    "etc", "ect", "v.v", "vv", "v_v", "vân_vân",
    
    # Nhóm token chung, dễ xuất hiện trong nhiều chuyên mục
    "mới", "lớn", "nhỏ", "cao", "thấp", "tốt", "quan_trọng",
    "đặc_biệt", "đáng_chú_ý", "đầu_tiên", "cuối_cùng",
    "phát_triển", "sử_dụng", "thực_hiện", "hỗ_trợ", "tạo", "tăng",
    "giảm", "đạt", "mang", "giúp", "gặp", "làm", "đưa", "nhận",
    "cung_cấp", "xây_dựng", "quản_lý", "cập_nhật"
}

punctuation_tokens = {
    ".", ",", "!", "?", ":", ";", "-", "--", "%", "/", "(", ")", "[", "]", "{", "}", '"', "'"
}

stopword_set = custom_stopwords.union(punctuation_tokens)

def remove_stopwords(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    tokens = text.split()
    filtered_tokens = []

    for token in tokens:
        token = token.strip()

        if not token:
            continue

        if token in stopword_set:
            continue

        if len(token) == 1 and not token.isalnum():
            continue

        filtered_tokens.append(token)

    return " ".join(filtered_tokens)

df_work["cluster_text_lexical"] = df_work["cluster_text_segmented"].apply(remove_stopwords)

print("Đã tạo xong cột cluster_text_lexical")
print("Kích thước dữ liệu hiện tại:", df_work.shape)
print("Số stopwords đang sử dụng:", len(stopword_set))

print("\nSo sánh dòng 0 trước và sau khi xóa stopwords:")
print("- Segmented :", df_work.loc[0, "cluster_text_segmented"][:250], "...")
print("- Lexical   :", df_work.loc[0, "cluster_text_lexical"][:250], "...")

display(
    df_work[["doc_id", "category_clean", "cluster_text_segmented", "cluster_text_lexical"]].head(3)
)

Đã tạo xong cột cluster_text_lexical
Kích thước dữ liệu hiện tại: (8727, 15)
Số stopwords đang sử dụng: 194

So sánh dòng 0 trước và sau khi xóa stopwords:
- Segmented : làn_sóng máy_tính lượng_tử sau cơn_sốt ai . quý 1 - 2026 , thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết trên các sàn chứng_khoán mỹ , được xem là tín_hiệu rõ_rệt cho thấy công_nghệ này đang có những bước_đi đầu_tiên trên hành_ ...
- Lexical   : làn_sóng máy_tính lượng_tử cơn_sốt ai thế_giới chứng_kiến loạt công_ty vi_tính lượng_tử liên_tiếp niêm_yết sàn chứng_khoán tín_hiệu rõ_rệt công_nghệ bước_đi hành_trình thương_mại_hóa đáng chú_ý giai_đoạn sàng_lọc công_nghệ thông_thường dòng tiền đổ_d ...


,doc_id,category_clean,cluster_text_segmented,cluster_text_lexical
0,1,cong_nghe,làn_sóng máy_tính lượng_tử sau cơn_sốt ai . qu...,làn_sóng máy_tính lượng_tử cơn_sốt ai thế_giới...
1,2,cong_nghe,meta đưa ai vào messenger trả_lời khách_hàng 2...,meta ai messenger trả_lời khách_hàng 24 khách_...
2,3,cong_nghe,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...,hà_tiên trí_tuệ nhân_tạo phục_vụ hành_chính cô...


### Nhận xét sau bước tinh chỉnh stopwords

Bộ stopwords trong notebook này không chỉ dựa trên danh sách từ dừng chung, mà còn được tinh chỉnh theo đặc thù của corpus báo chí. Một số token như `người`, `cho_biết`, `việt_nam`, `tp`, `hcm`, `đồng_thời`, `khiến` xuất hiện ở nhiều chuyên mục nhưng không giúp phân biệt chủ đề rõ ràng, nên được loại bỏ khỏi nhánh lexical.

Cách làm này giúp giảm nhiễu trong không gian đặc trưng TF-IDF/BM25 và phù hợp với mục tiêu gom cụm văn bản, vì trọng tâm là giữ lại các từ có khả năng thể hiện chủ đề, thay vì giữ những từ phổ biến nhưng ít giá trị phân biệt.

In [10]:
# Cell 13: tạo cột semantic_text và lưu file tiền xử lý cuối cùng của Notebook 2
# File output của cell này là đầu vào cho Notebook 03, 04 và 05.

# semantic_text dùng cho MiniLM:
# Giữ văn bản tự nhiên, không dùng bản đã bỏ stopword, không dùng bản đã tách từ.
# Chỉ lấy title + description + 300 từ đầu content để tránh văn bản quá dài bị MiniLM cắt tự động.

def build_semantic_text(row, max_content_words=300):
    title = str(row["title"]) if pd.notna(row["title"]) else ""
    description = str(row["description"]) if pd.notna(row["description"]) else ""
    content = str(row["content"]) if pd.notna(row["content"]) else ""

    # Lấy 300 từ đầu của content để giữ phần nội dung chính và tránh quá dài
    content_short = " ".join(content.split()[:max_content_words])

    # Ghép title + description + phần đầu content
    semantic_text = f"{title}. {description}. {content_short}"

    # Chuẩn hóa nhẹ, không bỏ stopword, không tách từ
    semantic_text = normalize_vietnamese_text(semantic_text)

    return semantic_text


df_work["cluster_text_semantic"] = df_work.apply(build_semantic_text, axis=1)

# Chuẩn hóa tên cột theo kịch bản mới
df_work["raw_text"] = df_work["cluster_text"]
df_work["clean_text"] = df_work["cluster_text_clean"]
df_work["segmented_text"] = df_work["cluster_text_segmented"]
df_work["lexical_text"] = df_work["cluster_text_lexical"]
df_work["semantic_text"] = df_work["cluster_text_semantic"]

# Nếu đã có word_count từ Notebook 1 thì giữ lại
# Nếu chưa có thì tự tạo từ raw_text
if "word_count" not in df_work.columns:
    df_work["word_count"] = df_work["raw_text"].astype(str).str.split().str.len()

final_columns = [
    "doc_id",
    "source",
    "category_clean",
    "title",
    "description",
    "content",
    "published_date",
    "url",
    "raw_text",
    "clean_text",
    "segmented_text",
    "lexical_text",
    "semantic_text",
    "word_count"
]

df_final_nlp = df_work[final_columns].copy()

# Lưu file tiền xử lý theo cấu trúc project mới
df_final_nlp.to_csv(OUTPUT_PREPROCESSED, index=False, encoding="utf-8-sig")

# Lưu bộ stopwords đã tinh chỉnh
with open(STOPWORDS_PATH, "w", encoding="utf-8") as f:
    for word in sorted(stopword_set):
        f.write(word + "\n")

print("--- TỔNG KẾT NOTEBOOK 2 ---")
print("Kích thước file NLP cuối cùng:", df_final_nlp.shape)

print("\nĐã lưu file tiền xử lý tại:")
print(OUTPUT_PREPROCESSED)

print("\nĐã lưu bộ refined stopwords tại:")
print(STOPWORDS_PATH)

print("\nCác cột đã lưu:")
print(df_final_nlp.columns.tolist())

print("\nKiểm tra số giá trị thiếu:")
display(df_final_nlp.isna().sum().to_frame("missing_count"))

print("\nSố dòng lexical_text rỗng:")
print((df_final_nlp["lexical_text"].fillna("").str.strip() == "").sum())

print("\nSố dòng semantic_text rỗng:")
print((df_final_nlp["semantic_text"].fillna("").str.strip() == "").sum())

print("\nThống kê độ dài semantic_text:")
display(df_final_nlp["semantic_text"].astype(str).str.split().str.len().describe().to_frame("semantic_word_count"))

display(df_final_nlp.head(3))

--- TỔNG KẾT NOTEBOOK 2 ---
Kích thước file NLP cuối cùng: (8727, 14)

Đã lưu file tiền xử lý tại:
D:\PROJECT\data\processed\news_preprocessed.csv

Đã lưu bộ refined stopwords tại:
D:\PROJECT\outputs\reports\refined_stopwords.txt

Các cột đã lưu:
['doc_id', 'source', 'category_clean', 'title', 'description', 'content', 'published_date', 'url', 'raw_text', 'clean_text', 'segmented_text', 'lexical_text', 'semantic_text', 'word_count']

Kiểm tra số giá trị thiếu:


,missing_count
doc_id,0
source,0
category_clean,0
title,0
description,0
content,0
published_date,0
url,0
raw_text,0
clean_text,0



Số dòng lexical_text rỗng:
0

Số dòng semantic_text rỗng:
0

Thống kê độ dài semantic_text:


,semantic_word_count
count,8727.000000
mean,348.291165
std,27.493439
min,80.000000
25%,345.000000
50%,353.000000
75%,361.000000
max,385.000000


,doc_id,source,category_clean,title,description,content,published_date,url,raw_text,clean_text,segmented_text,lexical_text,semantic_text,word_count
0,1,tuoitre,cong_nghe,Làn sóng máy tính lượng tử sau cơn sốt AI,"Quý 1-2026, thế giới chứng kiến loạt công ty v...","Đáng chú ý, khác với các giai đoạn sàng lọc cô...",2026-04-11T11:36:52+07:00,https://tuoitre.vn/lan-song-may-tinh-luong-tu-...,Làn sóng máy tính lượng tử sau cơn sốt AI . Qu...,làn sóng máy tính lượng tử sau cơn sốt ai . qu...,làn_sóng máy_tính lượng_tử sau cơn_sốt ai . qu...,làn_sóng máy_tính lượng_tử cơn_sốt ai thế_giới...,làn sóng máy tính lượng tử sau cơn sốt ai. quý...,1096
1,2,tuoitre,cong_nghe,Meta đưa AI vào Messenger trả lời khách hàng 24/7,Khách hàng nhắn tin lúc nửa đêm vẫn có người t...,Meta vừa giới thiệu trợ lý kinh doanh Business...,2026-04-11T07:11:28+07:00,https://tuoitre.vn/meta-dua-ai-vao-messenger-t...,Meta đưa AI vào Messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả lời khách hàng 2...,meta đưa ai vào messenger trả_lời khách_hàng 2...,meta ai messenger trả_lời khách_hàng 24 khách_...,meta đưa ai vào messenger trả lời khách hàng 2...,615
2,3,tuoitre,cong_nghe,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,"Ngày 10-4, UBND phường Hà Tiên (An Giang) ra m...","Phát biểu tại buổi lễ ra mắt, ông Trần Hải Quố...",2026-04-10T16:23:50+07:00,https://tuoitre.vn/ha-tien-su-dung-tri-tue-nha...,Hà Tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,hà tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,hà_tiên sử_dụng trí_tuệ nhân_tạo phục_vụ hành_...,hà_tiên trí_tuệ nhân_tạo phục_vụ hành_chính cô...,hà tiên sử dụng trí tuệ nhân tạo phục vụ hành ...,366
